### Import Libraries Step 1

In [38]:
import os 
from openai import OpenAI 
from dotenv import load_dotenv
from IPython.display import Markdown , display
import json

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")   

if OPENAI_API_KEY is None:
    raise Exception ("API Key is missing")
    
else:
   print(OPENAI_API_KEY[:8])

sk-proj-


### STEP 2: Setup Pushover

In [39]:
load_dotenv()

True

In [40]:

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

#Use this (in private!) to test that your keys have been loaded.
#print(pushover_user)
#print(pushover_token)

In [41]:
#Test Pushover
import requests

def send_notification(message: str):
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)
    

In [42]:
send_notification("Hello from Pushover 222!")

### Step 3: Describe Pushover as LLM Tool

In [43]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events or updates or complete a task that the user has asked you to do. The message parameter should contain the content of the notification you want to send.",
    "parameters": {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Stwp 4: Add pushover to list of tools of LLM

In [44]:
tools = [{"type":"function", "function":send_notification_function}]

### STEP 2a and 2b: Pushover

In [45]:
import random

#Simulates rolling a single six-sided die
def dice_roll():
    result = random.randint(1,6)
    return result

#Describe function for the LLM
roll_dice_function = {
    "name": "dice_roll",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die or simulate a random number between 1 and 6.",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

#Add function to list of tools of LLM
tools.append({"type":"function", "function":roll_dice_function})

### Step 5: Calling the tool from an LLM

In [46]:
def handle_tool_call(tool_calls):
    tool_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        #print(f"Calling function {function_name}") #For future debugging ;)

        #Route to the approriate function based on function_name
        if function_name == "send_notification":
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "dice_roll":
            content = f"Rolled: {dice_roll()}"
        #elif function_name == "insert_function_name_3":
        #    content = insert_function_name_3(args["message"])
        #....
        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_results.append(tool_call_result)

    return tool_results

In [49]:
client = OpenAI()
messages=[
      {"role":"user","content":"Please do two things:\n 1) I'd like to roll 2 dice, and\n 2) Send me a notification with the highest of the rolls"}
]
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages,
    tools=tools
)

message = response.choices[0].message

#Check if model wants to call a tool
while message.tool_calls:
    from pprint import pprint
    pprint(message.tool_calls)
    tool_result = handle_tool_call(message.tool_calls) #whole list of tool calls on purpose
    messages.append(message)
    messages.extend(tool_result) #Changed from append() to extend() when we switched to

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools #will add this in the future
    )
    message = response.choices[0].message

print(message.content)

[ChatCompletionMessageFunctionToolCall(id='call_jAFY5qpLZhnybBNdMd5LbW9i', function=Function(arguments='{}', name='dice_roll'), type='function'),
 ChatCompletionMessageFunctionToolCall(id='call_umVNWpilfgrR9ko0RdDcG52E', function=Function(arguments='{}', name='dice_roll'), type='function')]
[ChatCompletionMessageFunctionToolCall(id='call_FhDc1ajHi3xpB0YgNExCruLj', function=Function(arguments='{"message":"The highest roll of the two dice is 5."}', name='send_notification'), type='function')]
You rolled two dice with results 3 and 5. I have sent you a notification with the highest roll, which is 5.
